### EDA (Exploratory Data Analysis):

#### Import packages:

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import MinMaxScaler, normalize
from sklearn.metrics import silhouette_score, adjusted_rand_score, homogeneity_score
import plotly.express as px 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, DBSCAN


#### Load the dataset for EDA:

In [2]:
global_mobile_review_eda_df = pd.read_csv(r"C:\Users\jagad\Documents\my_classes\Tasks\mini_project_4-global_mobile_reviews\data\processed_data\Mobile_Reviews_Sentiment_before_encoding.csv")

print("\nData Shape: ")
print(global_mobile_review_eda_df.shape)
print("- "*50)

global_mobile_review_eda_df['year'] = global_mobile_review_eda_df['year'].astype(str)

print("\nInformation: ")
print(global_mobile_review_eda_df.info())
print("- "*50)

print("\nFirst 5 rows: ")
print(global_mobile_review_eda_df.head(5))
print("- "*50)



Data Shape: 
(50000, 20)
- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 

Information: 
<class 'pandas.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   age                  50000 non-null  int64  
 1   brand                50000 non-null  str    
 2   model                50000 non-null  str    
 3   price_usd            50000 non-null  float64
 4   rating               50000 non-null  int64  
 5   sentiment            50000 non-null  str    
 6   country              50000 non-null  str    
 7   language             50000 non-null  str    
 8   verified_purchase    50000 non-null  int64  
 9   battery_life_rating  50000 non-null  int64  
 10  camera_rating        50000 non-null  int64  
 11  performance_rating   50000 non-null  int64  
 12  design_rating        50000 non-null  int64  
 13  displ

### Visualize the sentments column:

In [3]:
sentiment_count = global_mobile_review_eda_df['sentiment'].value_counts().sort_index().reset_index()

print(sentiment_count)

sentiment_count.columns = ["sentiment", "count"]

fig = px.pie(
                sentiment_count,
                names="sentiment",
                values="count",
                title="Customer Sentiment Distribution",
                color="sentiment",
                color_discrete_map={
                    "Negative": "red",
                    "Neutral": "yellow",
                    "Positive": "green"
                }
)
fig.update_layout( 
                    height = 500,
                    width = 800,
                    title_x = 0.5
)
fig.update_traces( textinfo="percent+label" )
fig.show()


  sentiment  count
0  Negative  10656
1   Neutral  12035
2  Positive  27309


In [4]:
correlation_columns = ['price_usd','rating','battery_life_rating','camera_rating','performance_rating',
                       'design_rating','display_rating','helpful_votes']

correlation_matrix = global_mobile_review_eda_df[correlation_columns].corr()

print("\n Correlation Matrix:")
print(correlation_matrix)

correlation_heatmap_fig = px.imshow(correlation_matrix, 
                                                 title='Correlation Analysis for Gobal Mobile for Clustering Model',
                                                 color_continuous_scale="RDBu_r",
                                                 color_continuous_midpoint=0,
                                                 labels = dict(color="Correlation",fontsize = 10, fontweight = 'bold'),
                                                 text_auto='.2f',
                                                 aspect="equal"                              
                                                  )
correlation_heatmap_fig.update_layout(title_x = 0.30,
                                                   width=800,
                                                   height=600,
                                                   xaxis=dict(tickangle=-90),
                                                   coloraxis_colorbar=dict(
                                                   len=0.9,                      
                                                   yanchor="middle",             
                                                   y=0.5     )
                                                      )
correlation_heatmap_fig.show()

print("Correlation: ")
print(correlation_matrix['price_usd'].sort_values(ascending=False))


 Correlation Matrix:
                     price_usd    rating  battery_life_rating  camera_rating  \
price_usd             1.000000  0.000846             0.000999       0.001608   
rating                0.000846  1.000000             0.761990       0.759993   
battery_life_rating   0.000999  0.761990             1.000000       0.588938   
camera_rating         0.001608  0.759993             0.588938       1.000000   
performance_rating   -0.004727  0.755236             0.584341       0.579975   
design_rating         0.002682  0.755419             0.586143       0.580692   
display_rating       -0.000489  0.756719             0.584415       0.581909   
helpful_votes         0.003106  0.457125             0.355964       0.355280   

                     performance_rating  design_rating  display_rating  \
price_usd                     -0.004727       0.002682       -0.000489   
rating                         0.755236       0.755419        0.756719   
battery_life_rating            0.58

Correlation: 
price_usd              1.000000
helpful_votes          0.003106
design_rating          0.002682
camera_rating          0.001608
battery_life_rating    0.000999
rating                 0.000846
display_rating        -0.000489
performance_rating    -0.004727
Name: price_usd, dtype: float64


In [5]:
verified_purchase_pie_df = (
            global_mobile_review_eda_df.groupby("verified_purchase").size().reset_index(name="Count")
                            )

verified_purchase_pie_df["Label"] = verified_purchase_pie_df["verified_purchase"].map({
                                    1: "Verified purchase",
                                    0: "Not Verified purchase"
})
verified_purchase_pie_fig = px.pie(verified_purchase_pie_df,
                                    names="Label",
                                    values="Count",
                                    title="Overall Verified Purchase Distribution",
                                    hole=0.4  # Donut chart
)
verified_purchase_pie_fig.update_traces(textinfo="percent+label",
                                        textfont_size=14
)
verified_purchase_pie_fig.update_layout(
                                        width=700,
                                        height=500,
                                        title_x=0.25
)
verified_purchase_pie_fig.show()


In [6]:
# No of models per brand:
brand_model_counts = global_mobile_review_eda_df.groupby('brand')['model'].nunique().reset_index()

brand_model_counts_fig = px.bar(brand_model_counts, 
                                x='brand', 
                                y='model', 
                                title='Number of Models per Brand',
                                text='model',         # number on top of the bar
                                color='brand'         # each bar a unique color
)

brand_model_counts_fig.update_layout( xaxis_title = "Brand",
                                     yaxis_title = "No Of Models",
                                    width=900,
                                    height=500,
                                    title_x=0.5
)

brand_model_counts_fig.show()

In [7]:
country_model_counts = global_mobile_review_eda_df.groupby('country')['model'].nunique().reset_index()

country_model_counts_fig = px.bar(country_model_counts, 
                                x='country', 
                                y='model', 
                                title='Number of Models per Country',
                                text='model',         # number on top of the bar
                                color='country'         # each bar a unique color
)

country_model_counts_fig.update_layout( xaxis_title = "Country",
                                       yaxis_title = "No Of Models",
                                       width=1000,
                                       height=500,
                                      title_x=0.5
)

country_model_counts_fig.show()

In [8]:
brand_counts = global_mobile_review_eda_df['brand'].value_counts().reset_index()

brand_counts_fig = px.bar(
                            brand_counts, 
                            x='brand', 
                            y='count', 
                            title='Product Count by Brand',
                            text='count',      # Shows the number on the bar
                            color='brand'      # Gives each bar a unique color
)

brand_counts_fig.update_layout(  xaxis_title="Brand Name",
                                 yaxis_title="Count",
                                 width=900,
                                 height=500,
                                 title_x=0.5
)

brand_counts_fig.show()


In [9]:
country_counts = global_mobile_review_eda_df['country'].value_counts().reset_index()

country_counts_fig = px.bar(
                            country_counts, 
                            x='country', 
                            y='count', 
                            title='Product Count by Each Country',
                            text='count',        # Shows the number on the bar
                            color='country'      # Gives each bar a unique color
)

country_counts_fig.update_layout(  xaxis_title="Country Name",
                                 yaxis_title="Count",
                                 width=900,
                                 height=500,
                                 title_x=0.5
)

country_counts_fig.show()

In [10]:
brand_price_sum = global_mobile_review_eda_df.groupby('brand')['price_usd'].sum().reset_index()

brand_price_sum_fig = px.bar(
                                brand_price_sum, 
                                x='brand', 
                                y='price_usd', 
                                title='Overall Sales Value (Price USD) by Brand',
                                text='price_usd', # Display the sum on the bar
                                color='brand'
)

brand_price_sum_fig.update_traces(texttemplate='%{text:.3s}', textposition='outside')

brand_price_sum_fig.update_layout(  xaxis_title="Brand Name",
                                    yaxis_title="Total Amount (USD)",
                                    width=900,
                                    height=500,
                                    title_x=0.5
)


brand_price_sum_fig.show()


In [11]:
country_price_sum = global_mobile_review_eda_df.groupby('country')['price_usd'].sum().reset_index()

country_price_sum_fig = px.bar(
                                country_price_sum, 
                                x='country', 
                                y='price_usd', 
                                title='Overall Sales Value (Price USD) by Each Country',
                                text='price_usd', # Display the sum on the bar
                                color='country'
)

country_price_sum_fig.update_traces(texttemplate='%{text:.3s}', textposition='outside')

country_price_sum_fig.update_layout(  xaxis_title="Country Name",
                                    yaxis_title="Total Amount (USD)",
                                    width=900,
                                    height=500,
                                    title_x=0.5
)

# 5. Display the chart
country_price_sum_fig.show()

In [12]:
brand_year_price_sum = global_mobile_review_eda_df.groupby(['brand', 'year'])['price_usd'].sum().reset_index()

# 2. Create the plot
brand_price_sum_fig = px.bar(
                        brand_year_price_sum, 
                        x='brand', 
                        y='price_usd', 
                        color='year',           # This automatically creates a legend for years
                        barmode='group',        # 'group' makes the bars side-by-side; use 'stack' if you prefer stacked bars
                        title='Total Sales Value (Price USD) by Brand and Year',
                        text='price_usd'
)

# 3. Format the text labels
brand_price_sum_fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')

# 4. Update layout
brand_price_sum_fig.update_layout( 
                                    xaxis_title="Brand Name",
                                    yaxis_title="Total Amount (USD)",
                                    width=1100,
                                    height=500,
                                    title_x=0.5
)

brand_price_sum_fig.show()

In [13]:
country_year_price_sum = global_mobile_review_eda_df.groupby(['country', 'year'])['price_usd'].sum().reset_index()

# 2. Create the plot
country_price_sum_fig = px.bar(
                        country_year_price_sum, 
                        x='country', 
                        y='price_usd', 
                        color='year',           # This automatically creates a legend for years
                        barmode='group',        # 'group' makes the bars side-by-side; use 'stack' if you prefer stacked bars
                        title='Total Sales Value (Price USD) by Country and Year',
                        text='price_usd'
)

# 3. Format the text labels
country_price_sum_fig.update_traces(texttemplate='%{text:.2s}', textposition='outside')

# 4. Update layout
country_price_sum_fig.update_layout( 
                                    xaxis_title="Country Name",
                                    yaxis_title="Total Amount (USD)",
                                    width=1100,
                                    height=500,
                                    title_x=0.5
)

country_price_sum_fig.show()

In [14]:
top_10_models = global_mobile_review_eda_df['model'].value_counts().head(15).reset_index()

top_10_models_fig = px.bar(
                        top_10_models,
                        x='count',
                        y='model',
                        orientation='h', 
                        title='Top 15 Most Reviewed (Saled) Mobile Models',
                        text='count',
                        color_continuous_scale='blues'
)

top_10_models_fig.update_layout(
                    xaxis_title="Number of Reviews",
                    yaxis_title="Mobile Model",
                    width=1000,
                    height=600,
                    yaxis={'categoryorder':'total ascending'}, # Keeps the bar with the highest count at the top
                    title_x=0.5
)

top_10_models_fig.show()
